In [1]:
!pip freeze | grep scikit-learn

scikit-learn==1.5.0


In [2]:
!python -V

Python 3.10.13


In [1]:
import pickle
import pandas as pd

In [3]:
with open('model.bin', 'rb') as f_in:
    dv, model = pickle.load(f_in)

c:\Users\asus\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.5.0 when using version 1.3.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\asus\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\base.py:348: InconsistentVersionWarning: Trying to unpickle estimator LinearRegression from version 1.5.0 when using version 1.3.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [2]:
categorical = ['PULocationID', 'DOLocationID']

def read_data(filename):
    df = pd.read_parquet(filename)
    
    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df['duration'] = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)].copy()

    df[categorical] = df[categorical].fillna(-1).astype('int').astype('str')
    
    return df

In [4]:
# df = read_data('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_????-??.parquet')
df = read_data('data/yellow_tripdata_2023-03.parquet')

In [5]:
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,duration
0,2,2023-03-01 00:06:43,2023-03-01 00:16:43,1.0,0.00,1.0,N,238,42,2,8.6,1.0,0.5,0.00,0.0,1.0,11.10,0.0,0.00,10.000000
1,2,2023-03-01 00:08:25,2023-03-01 00:39:30,2.0,12.40,1.0,N,138,231,1,52.7,6.0,0.5,12.54,0.0,1.0,76.49,2.5,1.25,31.083333
2,1,2023-03-01 00:15:04,2023-03-01 00:29:26,0.0,3.30,1.0,N,140,186,1,18.4,3.5,0.5,4.65,0.0,1.0,28.05,2.5,0.00,14.366667
3,1,2023-03-01 00:49:37,2023-03-01 01:01:05,1.0,2.90,1.0,N,140,43,1,15.6,3.5,0.5,4.10,0.0,1.0,24.70,2.5,0.00,11.466667
4,2,2023-03-01 00:08:04,2023-03-01 00:11:06,1.0,1.23,1.0,N,79,137,1,7.2,1.0,0.5,2.44,0.0,1.0,14.64,2.5,0.00,3.033333


In [6]:
dicts = df[categorical].to_dict(orient='records')
X_val = dv.transform(dicts)
y_pred = model.predict(X_val)

## Question 1 Standard deviation 

Answer: 6.25

In [7]:
import numpy as np
std = np.std(y_pred)
print(f'Standard deviation of predictions: {std:.2f}')

Standard deviation of predictions: 6.25


## Question 2 Preparing the output

Answer: 65.46 MB or close one 66 MB

In [ ]:
year = 2023
month = 3

In [9]:
df['ride_id'] = f'{year:04d}/{month:02d}_' + df.index.astype('str')

In [10]:
#prepare for df_results with only ride_id and prediction
df_result = pd.DataFrame()

In [11]:
df_result['ride_id']  = df['ride_id']
df_result['predicted_duration']  = y_pred

In [12]:
!mkdir output
output_file = 'output/predict_yellow_2023_03_v2.parquet'

In [13]:
df_result.to_parquet(
    output_file,
    engine='pyarrow',
    compression=None,
    index=False
)

In [14]:
import os
file_path = os.path.abspath(output_file)
file_size = os.path.getsize(file_path) / (1024 * 1024)

print(f'File {file_path} is created with size {file_size:.2f} MB')

File d:\github\mlops_2025\module_4\homework\output\predict_yellow_2023_03_v2.parquet is created with size 65.46 MB


## Question 3 Create the scoring script

Now let's turn the notebook into a script.

Which command you need to execute for that?

Answer: `jupyter nbconvert --to script {hw_answer_04}.ipynb`

## Question 4 virtual environtment


What's the first hash for the Scikit-Learn dependency?

Answer: "sha256:057b991ac64b3e75c9c04b5f9395eaf19a6179244c089afdebaad98264bff37c"

## Question 5 Parametrize the script

when trying to run the script make sure to add the parameter. So it should like this:
`python mean_predict.py 2023 04`

Answer: Mean predicted duration:  14.292282936862449

## Question 6 Docker Container

Since I use CLI arguments for year and month. So, when I need to use docker we can give command like this

`docker run my-predict-app 2023 04`

because it same with runs 

`python mean_predict.py 2023 04`

Answer:

